In [1]:
import pandas as pd
import glob
import numpy as np
import warnings
import math
from src.dataset.participant_preprocessing import ParticipantData
from src.utils.functions import mkdir_if_not_exists
from src.utils.logger import log
from logging import INFO
warnings.filterwarnings("ignore")

In [2]:
files = glob.glob(f'dataset/pecanstreet/15min/austin/california/[0-9]*.csv', recursive=True)
for f in files:
    print(pd.read_csv(f).shape)

In [3]:
locations = ['austin', 'newyork', 'california', 'puertorico']

In [4]:
def _divide_train_test_set(location):
    files = glob.glob(f'dataset/pecanstreet/15min/{location}/*.csv', recursive=True)
    mkdir_if_not_exists(f'dataset/pecanstreet/15min/{location}/train')
    mkdir_if_not_exists(f'dataset/pecanstreet/15min/{location}/test')
    filepath = f'dataset/pecanstreet/15min/{location}/'
    num_lags=96
    for file in files:
        df, cid = ParticipantData.preprocess_readings(file, l)
        data_description = df.describe().transpose()
        log(INFO, f"cid: {cid} | Count: {data_description['count']['consumption']} | Mean: {data_description['mean']['consumption']} | Median: {data_description['50%']['consumption']} | Std: {data_description['std']['consumption']}")
        test_df = df.tail(num_lags * 2)
        temp_df = test_df.head(num_lags)
        
        i_end_train = temp_df.index[-1]
        train_df = df.iloc[: i_end_train + 1]
        
        train_df.to_csv(f"dataset/pecanstreet/15min/{location}/train/{cid}.csv", index_label=False)
        test_df.to_csv(f"dataset/pecanstreet/15min/{location}/test/{cid}.csv", index_label=False)
        log(INFO, f'dataset/pecanstreet/15min/{location}/train/{cid}.csv')
        log(INFO, f'dataset/pecanstreet/15min/{location}/test/{cid}.csv')



In [5]:
for l in locations:
    _divide_train_test_set(l)

INFO logger 2026-05-11 09:59:30,620 | participant_preprocessing.py:49 | Preprocessing readings from 2818
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34980/34980 [00:31<00:00, 1125.51it/s]
INFO logger 2026-05-11 10:00:03,047 | participant_preprocessing.py:86 | 2818 data loaded with shape: (34980, 46)
INFO logger 2026-05-11 10:00:03,123 | 3829228450.py:10 | cid: 2818 | Count: 34980.0 | Mean: 0.6697529159519726 | Median: 0.456 | Std: 1.5261952069439708
INFO logger 2026-05-11 10:00:04,366 | 3829228450.py:19 | dataset/pecanstreet/15min/austin/train/2818.csv
INFO logger 2026-05-11 10:00:04,367 | 3829228450.py:20 | dataset/pecanstreet/15min/austin/test/2818.csv
INFO logger 2026-05-11 10:00:05,195 | participant_preprocessing.py:49 | Preprocessing readings from 4767
100%|████████████████████████████████████████

Exception: 

In [66]:
def _gen_community_dataset(location):
    train_files = glob.glob(f'dataset/pecanstreet/15min/{location}/train/*[0-9].csv', recursive=True)
    test_files = glob.glob(f'dataset/pecanstreet/15min/{location}/test/*[0-9].csv', recursive=True)
    log(INFO, f"Number of train files: {len(train_files)} | Number of test files: {len(test_files)}")
    if f'dataset/pecanstreet/15min/{location}/train/full_dataset.csv' in train_files:
        raise KeyboardInterrupt(f"full_dataset.csv must not be included in the train file for the community")
    train_sum_df = pd.read_csv(train_files[0])
    test_sum_df = pd.read_csv(test_files[0])
    test_sum_df.reset_index(drop=True, inplace=True)
    train_sum_df.fillna(value=0, inplace=True)
    test_sum_df.fillna(value=0, inplace=True)
    for train_file, test_file in zip(train_files, test_files):
        train_df = pd.read_csv(train_file)
        test_df = pd.read_csv(test_file)
        train_df.reset_index(drop=True, inplace=True)
        test_df.reset_index(drop=True, inplace=True)
        train_df.fillna(value=0, inplace=True)
        test_df.fillna(value=0, inplace=True)
        
        # _df = pd.concat([train_df, test_df[96:]]).reset_index(drop=True)
        
        min_length = min(len(train_df['consumption']), len(train_df['consumption']))
        train_sum_df['consumption'][:min_length] += train_df['consumption'][:min_length]
        test_sum_df['consumption'] += test_df['consumption']
        
        train_sum_df['generation'][:min_length] += train_df['generation'][:min_length]
        test_sum_df['generation'] += test_df['generation']
                
    train_sum_df['prev_consumption'] = train_sum_df.shift(1)['consumption']
    train_sum_df['consumption_change'] = train_sum_df.apply(
        lambda row: 0 if np.isnan(row.prev_consumption) else row.consumption - row.prev_consumption, axis=1)
    test_sum_df['prev_consumption'] = test_sum_df.shift(1)['consumption']
    test_sum_df['consumption_change'] = test_sum_df.apply(
        lambda row: 0 if np.isnan(row.prev_consumption) else row.consumption - row.prev_consumption, axis=1)
    
    
    train_sum_df['cid'] = 'community'
    test_sum_df['cid'] = 'community'
    train_sum_df.to_csv(f'dataset/pecanstreet/15min/{location}/train/community.csv', index_label=False)
    test_sum_df.to_csv(f'dataset/pecanstreet/15min/{location}/test/community.csv', index_label=False)
    log(INFO, f'{location} community data saved on dataset/pecanstreet/15min/{location}/train [test]/community.csv')



In [67]:
for l in locations:
    _gen_community_dataset(l)

INFO logger 2025-02-16 09:49:23,745 | 1614535685.py:4 | Number of train files: 25 | Number of test files: 25
INFO logger 2025-02-16 09:49:27,567 | 1614535685.py:41 | austin community data saved on dataset/pecanstreet/15min/austin/train [test]/community.csv
INFO logger 2025-02-16 09:49:27,571 | 1614535685.py:4 | Number of train files: 25 | Number of test files: 25
INFO logger 2025-02-16 09:49:29,520 | 1614535685.py:41 | newyork community data saved on dataset/pecanstreet/15min/newyork/train [test]/community.csv
INFO logger 2025-02-16 09:49:29,522 | 1614535685.py:4 | Number of train files: 23 | Number of test files: 23
INFO logger 2025-02-16 09:49:32,863 | 1614535685.py:41 | california community data saved on dataset/pecanstreet/15min/california/train [test]/community.csv
INFO logger 2025-02-16 09:49:32,866 | 1614535685.py:4 | Number of train files: 25 | Number of test files: 25
INFO logger 2025-02-16 09:49:33,737 | 1614535685.py:41 | puertorico community data saved on dataset/pecanstree

In [76]:
pd.read_csv(f'dataset/pecanstreet/15min/austin/train/community.csv').describe().transpose()

,count,mean,std,min,25%,50%,75%,max
leg1v,34888.0,122.730825,3.782966,0.000000,1.224040e+02,122.896000,123.312000,125.026000
leg2v,34888.0,122.650518,3.821541,0.000000,1.222720e+02,122.843000,123.362000,125.168000
generation,34888.0,14.978868,17.023088,0.000000,1.979750e+00,9.164500,21.690250,76.435000
consumption,34888.0,22.535614,22.135903,-57.484000,1.198300e+01,21.578000,35.224250,108.262000
year,34888.0,2018.000000,0.000000,2018.000000,2.018000e+03,2018.000000,2018.000000,2018.000000
month,34888.0,6.517313,3.443001,1.000000,4.000000e+00,7.000000,10.000000,12.000000
day,34888.0,15.693534,8.776853,1.000000,8.000000e+00,16.000000,23.000000,31.000000
hour,34888.0,11.498739,6.919182,0.000000,6.000000e+00,11.000000,17.000000,23.000000
minute,34888.0,22.500000,16.770750,0.000000,1.125000e+01,22.500000,33.750000,45.000000
second,34888.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000


In [52]:
train_files = glob.glob(f'dataset/pecanstreet/15min/austin/train/*[0-9].csv', recursive=True)
test_files = glob.glob(f'dataset/pecanstreet/15min/austin/test/*[0-9].csv', recursive=True)
train_sum_df = pd.read_csv(train_files[0])
test_sum_df = pd.read_csv(test_files[0])
test_sum_df.reset_index(drop=True, inplace=True)
train_sum_df.fillna(value=0, inplace=True)
test_sum_df.fillna(value=0, inplace=True)
# init_df = pd.concat([train_sum_df, test_sum_df[96:]]).reset_index(drop=True)

In [55]:
len(train_files), len(test_files)

(25, 25)

In [54]:
test_files

['dataset/pecanstreet/15min/austin/test/8156.csv',
 'dataset/pecanstreet/15min/austin/test/2335.csv',
 'dataset/pecanstreet/15min/austin/test/9922.csv',
 'dataset/pecanstreet/15min/austin/test/3039.csv',
 'dataset/pecanstreet/15min/austin/test/4031.csv',
 'dataset/pecanstreet/15min/austin/test/8386.csv',
 'dataset/pecanstreet/15min/austin/test/7951.csv',
 'dataset/pecanstreet/15min/austin/test/3538.csv',
 'dataset/pecanstreet/15min/austin/test/9160.csv',
 'dataset/pecanstreet/15min/austin/test/5746.csv',
 'dataset/pecanstreet/15min/austin/test/1642.csv',
 'dataset/pecanstreet/15min/austin/test/6139.csv',
 'dataset/pecanstreet/15min/austin/test/2361.csv',
 'dataset/pecanstreet/15min/austin/test/3456.csv',
 'dataset/pecanstreet/15min/austin/test/9019.csv',
 'dataset/pecanstreet/15min/austin/test/661.csv',
 'dataset/pecanstreet/15min/austin/test/2818.csv',
 'dataset/pecanstreet/15min/austin/test/7800.csv',
 'dataset/pecanstreet/15min/austin/test/7536.csv',
 'dataset/pecanstreet/15min/aust

In [47]:
for train_file, test_file in zip(train_files[1:], test_files[1:]):
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    train_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)
    train_df.fillna(value=0, inplace=True)
    test_df.fillna(value=0, inplace=True)
    
    # _df = pd.concat([train_df, test_df[96:]]).reset_index(drop=True)
    
    min_length = min(len(train_df['consumption']), len(train_df['consumption']))
    train_sum_df['consumption'][:min_length] += train_df['consumption'][:min_length]
    test_sum_df['consumption'] += test_df['consumption']
    
    train_sum_df['generation'][:min_length] += train_df['generation'][:min_length]
    test_sum_df['generation'] += test_df['generation']
    
    
train_sum_df['prev_consumption'] = train_sum_df.shift(1)['consumption']
train_sum_df['consumption_change'] = train_sum_df.apply(
    lambda row: 0 if np.isnan(row.prev_consumption) else row.consumption - row.prev_consumption, axis=1)
test_sum_df['prev_consumption'] = test_sum_df.shift(1)['consumption']
test_sum_df['consumption_change'] = test_sum_df.apply(
    lambda row: 0 if np.isnan(row.prev_consumption) else row.consumption - row.prev_consumption, axis=1)


train_sum_df['cid'] = 'community'
test_sum_df['cid'] = 'community'

In [51]:
train_files

['dataset/pecanstreet/15min/austin/train/8156.csv',
 'dataset/pecanstreet/15min/austin/train/2335.csv',
 'dataset/pecanstreet/15min/austin/train/9922.csv',
 'dataset/pecanstreet/15min/austin/train/3039.csv',
 'dataset/pecanstreet/15min/austin/train/4031.csv',
 'dataset/pecanstreet/15min/austin/train/8386.csv',
 'dataset/pecanstreet/15min/austin/train/7951.csv',
 'dataset/pecanstreet/15min/austin/train/3538.csv',
 'dataset/pecanstreet/15min/austin/train/9160.csv',
 'dataset/pecanstreet/15min/austin/train/full_dataset.csv',
 'dataset/pecanstreet/15min/austin/train/5746.csv',
 'dataset/pecanstreet/15min/austin/train/1642.csv',
 'dataset/pecanstreet/15min/austin/train/6139.csv',
 'dataset/pecanstreet/15min/austin/train/2361.csv',
 'dataset/pecanstreet/15min/austin/train/3456.csv',
 'dataset/pecanstreet/15min/austin/train/9019.csv',
 'dataset/pecanstreet/15min/austin/train/661.csv',
 'dataset/pecanstreet/15min/austin/train/2818.csv',
 'dataset/pecanstreet/15min/austin/train/7800.csv',
 'dat

In [ ]:
mkdir_if_not_exists(f'dataset/pecanstreet/15min/train/')
mkdir_if_not_exists(f'dataset/pecanstreet/15min/test/')
train_sum_df.to_csv(f'dataset/pecanstreet/15min/train/community.csv', index_label=False)
test_sum_df.to_csv(f'dataset/pecanstreet/15min/test/community.csv', index_label=False)

In [ ]:
pd.read_csv(f'dataset/pecanstreet/15min/test/community.csv').describe().transpose()

In [ ]:
sun_df

In [ ]:
df

In [ ]:
temp_df = df.loc[df['Date'] < '2018-12-31']
temp_df

In [ ]:
temp_df.tail(96)

In [ ]:
df.tail()

In [ ]:
df.loc[(df['Date'] >= '2018-12-31') & (df['Date'] < '2019-01-01')]